In [48]:
import pandas as pd
import numpy as np
import import_ipynb
from utils import tokenize, apply_SMOTE, gen_word_embedding
from hypermodels import LSTMHyperModel, GRUHyperModel, CNNHyperModel, SBERTHyperModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import pickle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

import keras_tuner as kt
from keras_tuner import BayesianOptimization
from tensorflow.keras.callbacks import EarlyStopping


import wandb
from wandb.integration.keras import WandbCallback

from sentence_transformers import SentenceTransformer

from config import modelname, wandb_API_KEY

In [37]:
df = pd.read_csv("../data/sententence_data.csv")

In [38]:
df['Category'] =  df['entailment_AB'] + '_' + df['entailment_BA']
X = df[['sentence_A', 'sentence_B']]
y = df['Category']

In [ ]:
# modelname = 'SBERT'

### Encode the Target Column

In [39]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
y = to_categorical(y)

with open("../data/label_encoder_classes.pkl", 'wb') as f:
    pickle.dump(label_encoder.classes_, f)

In [40]:
def pre_process(modelname):
    if modelname != 'SBERT':
        MAX_NUM_WORDS = 100000
        MAX_SEQUENCE_LENGTH = 50
        EMBEDDING_DIM = 300

        word_index, padded_A, padded_B = tokenize(df, MAX_SEQUENCE_LENGTH, MAX_NUM_WORDS)

        X_combined = np.hstack([padded_A, padded_B])
        X_resampled, y_resampled = apply_SMOTE(X_combined, y)

        X_A_resampled = X_resampled[:, :MAX_SEQUENCE_LENGTH]
        X_B_resampled = X_resampled[:, MAX_SEQUENCE_LENGTH:]

        embedding_matrix = gen_word_embedding(EMBEDDING_DIM, MAX_NUM_WORDS,word_index)

        X_trainA, X_testA, X_trainB, X_testB, y_train, y_test = train_test_split(X_A_resampled, X_B_resampled, y_resampled, test_size=0.2, random_state=42)

        if modelname == 'LSTM':
            selected_model =  LSTMHyperModel(embedding_matrix=embedding_matrix, 
                              MAX_NUM_WORDS=MAX_NUM_WORDS,
                              EMBEDDING_DIM = EMBEDDING_DIM, 
                              MAX_SEQUENCE_LENGTH = MAX_SEQUENCE_LENGTH)
        
        elif modelname == 'GRU':
            selected_model = GRUHyperModel(embedding_matrix=embedding_matrix, 
                              MAX_NUM_WORDS=MAX_NUM_WORDS,
                              EMBEDDING_DIM = EMBEDDING_DIM, 
                              MAX_SEQUENCE_LENGTH = MAX_SEQUENCE_LENGTH)
        
        elif modelname == 'CNN':
            selected_model = CNNHyperModel(embedding_matrix=embedding_matrix, 
                              MAX_NUM_WORDS=MAX_NUM_WORDS,
                              EMBEDDING_DIM = EMBEDDING_DIM, 
                              MAX_SEQUENCE_LENGTH = MAX_SEQUENCE_LENGTH)
        

    else:
        sbert_model_name = 'all-mpnet-base-v2'
        sbert_model = SentenceTransformer(sbert_model_name)

        embeddings_A = sbert_model.encode(df['sentence_A'])
        embeddings_B = sbert_model.encode(df['sentence_B'])
        EMBEDDING_DIM = embeddings_A.shape[1]

        X_combined = np.hstack([embeddings_A, embeddings_B])
        X_resampled, y_resampled = apply_SMOTE(X_combined, y)

        X_A_resampled = X_resampled[:, :EMBEDDING_DIM]
        X_B_resampled = X_resampled[:, EMBEDDING_DIM:]

        X_trainA, X_testA, X_trainB, X_testB, y_train, y_test = train_test_split(X_A_resampled, X_B_resampled, y_resampled, test_size=0.2, random_state=42)
        selected_model = SBERTHyperModel(EMBEDDING_DIM = EMBEDDING_DIM)

    return selected_model, X_trainA, X_testA, X_trainB, X_testB, y_train, y_test

In [41]:
selected_model, X_trainA, X_testA, X_trainB, X_testB, y_train, y_test = pre_process(modelname)

In [42]:
wandb.login(key=wandb_API_KEY)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [49]:
class MyTuner(kt.Tuner):
    def run_trial(self, trial, X_input, y_train, batch_size, epochs, objective):
        hp = trial.hyperparameters
        objective_name_str = objective

        ## create the model with the current trial hyperparameters
        model = self.hypermodel.build(hp)

        ## Initiates new run for each trial on the dashboard of Weights & Biases
        run = wandb.init(project="entailment_classifier_LSTM", config=hp.values)
        print("I am running")

        history = model.fit(X_input,
                  y_train,
                  batch_size=batch_size,
                  epochs=epochs,
                  validation_split=0.1,
                  callbacks=[WandbCallback()])
        
        training_loss = history.history['loss'][-1] 

        
        self.oracle.update_trial(trial.trial_id, {objective_name_str:training_loss})

        run.finish()

### Hyper parameter Tuning

In [50]:
objective = 'loss' 

'''tuner = BayesianOptimization(
    lstm_model,
    objective='val_accuracy',
    max_trials=30,  # Number of different hyperparameter combinations to try
    directory='hp_tuning_dir',
    project_name='entailment_classifier'
)
'''
tuner = MyTuner(
      oracle=kt.oracles.BayesianOptimizationOracle(
          objective=objective,
          max_trials=4),
      hypermodel=selected_model,
      directory='../hp_tuning_dir')

tuner.search_space_summary()

Reloading Tuner from ../hp_tuning_dir\untitled_project\tuner0.json
Search space summary
Default search space size: 3
first_dense_units (Int)
{'default': None, 'conditions': [], 'min_value': 100, 'max_value': 150, 'step': 10, 'sampling': 'linear'}
dropout_1 (Float)
{'default': 0.25, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [51]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [61]:
''' tuner.search(
    X_input= [X_trainA, X_trainB],
    y_train=y_train,
    epochs=5,  # Reduced for faster tuning
    validation_data=([X_testA, X_testB], y_test),    
    batch_size=16,  # Can be larger with SBERT as we don't need to process sequences
    #callbacks=[early_stopping],
    objective=objective
)
'''

tuner.search(
    [X_trainA, X_trainB],
    #X_input= [X_trainA, X_trainB],
    y_train=y_train,
    epochs=5,  # Reduced for faster tuning
    #validation_data=([X_testA, X_testB], y_test),    
    batch_size=16,  # Can be larger with SBERT as we don't need to process sequences
    #callbacks=[early_stopping],
    objective=objective
)



I am running
Epoch 1/5


Traceback (most recent call last):
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\renju\AppData\Local\Temp\ipykernel_18460\4265931380.py", line 13, in run_trial
    history = model.fit(X_input,
              ^^^^^^^^^^^^^^^^^^
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "c:\Users\re

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\renju\AppData\Local\Temp\ipykernel_18460\4265931380.py", line 13, in run_trial
    history = model.fit(X_input,
              ^^^^^^^^^^^^^^^^^^
  File "c:\Users\renju\Documents\Renju\AI ML\Projects\DL-Projects\Textual Entailment\.venv13\Lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler

### Get the best model hyper parameters and train the model

In [55]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

In [56]:
run = wandb.init(project = 'Textual Entailment')

In [57]:
checkpoint_filepath = '../models/' + modelname + '.keras'
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)


In [58]:
history = best_model.fit(
    [X_trainA, X_trainB],
    y_train,
    validation_data=([X_testA, X_testB], y_test),
    batch_size=32,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint
               ],
    verbose=1
)




Epoch 1/10
872/874 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5548 - loss: 1.2524
Epoch 1: val_accuracy improved from -inf to 0.79954, saving model to ../models/SBERT.keras
874/874 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.5551 - loss: 1.2513 - val_accuracy: 0.7995 - val_loss: 0.5616
Epoch 2/10
867/874 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7972 - loss: 0.5714
Epoch 2: val_accuracy improved from 0.79954 to 0.86422, saving model to ../models/SBERT.keras
874/874 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7972 - loss: 0.5713 - val_accuracy: 0.8642 - val_loss: 0.3947
Epoch 3/10
869/874 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8425 - loss: 0.4586
Epoch 3: val_accuracy improved from 0.86422 to 0.90199, saving model to ../models/SBERT.keras
874/874 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8425 - loss: 0.4585 - val_accuracy: 0.9020 - val_loss: 0.3092
Epoch 4/10
872/874 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8703 - loss: 0.3794
Epoch 4: val_accuracy impr